# PDF Exploration & Structure Analysis
## Industrial Equipment Data Extraction Project

This notebook systematically answers the following questions about the source PDFs:

| # | Question |
|---|----------|
| 1 | Are the PDFs text-based or scanned? |
| 2 | How many pages does each have? |
| 3 | What text can we extract? |
| 4 | Are there tables? |
| 5 | How is equipment information organized? |
| 6 | Are the two PDFs structured similarly? |
| 7 | What fields do we need to extract? |

**Source PDFs:**
- `AUSTCOLD.pdf`
- `MYCOM Operating and Maintenance Manual Refrigeration Unit (1).pdf`

**Libraries used:** `pdfplumber`, `pypdfium2`, `pdfminer.six` (all pre-installed)

---
## 0. Setup & Imports

In [2]:
import os
import re
import sys
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import pdfplumber
import pypdfium2 as pdfium

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path(os.getcwd()).parent
DATA_DIR     = PROJECT_ROOT / 'data' / 'raw'

PDF_AUSTCOLD = DATA_DIR / 'AUSTCOLD.pdf'
PDF_MYCOM    = DATA_DIR / 'MYCOM Operating and Maintenance Manual Refrigeration Unit (1).pdf'

PDFS = {
    'AUSTCOLD': PDF_AUSTCOLD,
    'MYCOM':    PDF_MYCOM,
}

print('Library versions:')
import pdfplumber as _pp
print(f'  pdfplumber  : {_pp.__version__}')
print(f'  pypdfium2   : {pdfium.__version__}')
print()
for name, path in PDFS.items():
    size_mb = path.stat().st_size / 1_048_576
    status = '✅' if path.exists() else '❌ MISSING'
    print(f'{status}  {name}: {path.name}  ({size_mb:.1f} MB)')

ModuleNotFoundError: No module named 'pdfplumber'

---
## Question 1 — Are the PDFs text-based or scanned?

**Method:** We compare the number of extractable characters per page.
- **< 20 characters** on a page → treated as image-only (scanned)
- We use **pypdfium2** for fast per-page text extraction across the whole document.

In [ ]:
TEXT_CHAR_THRESHOLD = 20  # chars/page below which we call it 'scanned'

def classify_pages(pdf_path: Path, sample_size: int = 30):
    """
    Open with pypdfium2, sample pages evenly, classify each as text or scanned.
    Returns: (total_pages, list[dict], overall_label)
    """
    doc = pdfium.PdfDocument(str(pdf_path))
    total_pages = len(doc)

    # Evenly spaced sample
    step = max(1, total_pages // sample_size)
    indices = list(range(0, total_pages, step))[:sample_size]

    results = []
    for i in indices:
        page = doc[i]
        textpage = page.get_textpage()
        text = textpage.get_text_range().strip()
        chars = len(text)
        results.append({
            'page':       i + 1,
            'chars':      chars,
            'is_scanned': chars < TEXT_CHAR_THRESHOLD,
        })
    doc.close()

    scanned_count = sum(r['is_scanned'] for r in results)
    ratio = scanned_count / len(results)
    if ratio > 0.5:
        overall = 'SCANNED'
    elif ratio > 0.1:
        overall = 'MIXED'
    else:
        overall = 'TEXT-BASED'
    return total_pages, results, overall, ratio


print('Classifying pages …\n')
print('=' * 65)
print(f"{'PDF':<12} {'Total Pages':>12} {'Sampled':>8} {'Scanned%':>9}  Classification")
print('=' * 65)

page_classifications = {}
for name, path in PDFS.items():
    total, page_data, overall, ratio = classify_pages(path)
    page_classifications[name] = (total, page_data, overall)
    print(f'{name:<12} {total:>12,} {len(page_data):>8} {ratio*100:>8.1f}%  {overall}')

print('=' * 65)

In [ ]:
# Per-page detail table
for name, (total, page_data, overall) in page_classifications.items():
    print(f"\n{'─'*55}")
    print(f'  {name}  →  {overall}  ({total:,} pages total)')
    print(f"{'─'*55}")
    print(f"  {'Page':>5}  {'Chars':>8}  Status")
    for r in page_data:
        icon = '🔴 SCANNED' if r['is_scanned'] else '🟢 TEXT'
        print(f"  {r['page']:>5}  {r['chars']:>8,}  {icon}")

---
## Question 2 — How many pages does each PDF have?

In [ ]:
print('PDF Summary')
print('=' * 50)
print(f"  {'Name':<14} {'Pages':>8}  {'Size (MB)':>10}")
print('  ' + '─' * 36)
for name, path in PDFS.items():
    total, _, _ = page_classifications[name]
    mb = path.stat().st_size / 1_048_576
    print(f"  {name:<14} {total:>8,}  {mb:>10.1f}")
print('=' * 50)

---
## Question 3 — What text can we extract?

We sample pages from the beginning, middle, and end of each document and inspect the raw extracted text.

In [ ]:
def extract_text_sample(pdf_path: Path, page_indices: list[int]) -> dict:
    """Extract text from specified 0-indexed pages using pypdfium2."""
    doc = pdfium.PdfDocument(str(pdf_path))
    out = {}
    for i in page_indices:
        if 0 <= i < len(doc):
            tp = doc[i].get_textpage()
            out[i + 1] = tp.get_text_range().strip()
    doc.close()
    return out


PREVIEW_CHARS = 900

for name, path in PDFS.items():
    total, _, _ = page_classifications[name]
    indices = sorted(set([
        0, 1, 2, 3, 4,
        total // 4,
        total // 2,
        3 * total // 4,
        total - 1,
    ]))
    texts = extract_text_sample(path, indices)

    print(f"\n{'━'*65}")
    print(f'  📄 {name}  —  Text Samples')
    print(f"{'━'*65}")
    for pg, txt in texts.items():
        preview = txt[:PREVIEW_CHARS].replace('\n', ' ↵ ')
        print(f"\n  ── Page {pg:>4}  ({len(txt):,} chars) ──")
        print(f'  {preview}')
        if len(txt) > PREVIEW_CHARS:
            print(f'  … [{len(txt) - PREVIEW_CHARS:,} more chars] …')

In [ ]:
# Text quality metrics across the first 60 pages
def text_quality_metrics(pdf_path: Path, max_pages: int = 60) -> dict:
    doc = pdfium.PdfDocument(str(pdf_path))
    n = min(len(doc), max_pages)
    total_chars, total_words, blank_pages, text_pages = 0, 0, 0, 0

    for i in range(n):
        txt = doc[i].get_textpage().get_text_range().strip()
        c = len(txt)
        w = len(txt.split())
        total_chars += c
        total_words += w
        if c < TEXT_CHAR_THRESHOLD:
            blank_pages += 1
        else:
            text_pages += 1
    doc.close()

    return {
        'pages_scanned':              n,
        'text_pages':                 text_pages,
        'blank_or_image_pages':       blank_pages,
        'avg_chars_per_text_page':    round(total_chars / max(text_pages, 1)),
        'avg_words_per_text_page':    round(total_words / max(text_pages, 1)),
        'total_words_sampled':        total_words,
        'total_chars_sampled':        total_chars,
    }


print('Text Quality Metrics  (first 60 pages of each PDF)')
print('=' * 60)
for name, path in PDFS.items():
    m = text_quality_metrics(path)
    print(f'\n  {name}')
    for k, v in m.items():
        print(f'    {k:<35} {v:>10,}')
print('=' * 60)

---
## Question 4 — Are there tables?

**pdfplumber** is the best open-source tool for detecting tables in text-based PDFs — it analyses line geometry to identify cell boundaries.

In [ ]:
def find_tables(pdf_path: Path, max_pages: int = 80) -> list[dict]:
    """Use pdfplumber to detect tables and return metadata + sample rows."""
    found = []
    with pdfplumber.open(str(pdf_path)) as pdf:
        n = min(len(pdf.pages), max_pages)
        for i in range(n):
            tables = pdf.pages[i].extract_tables()
            for t_idx, table in enumerate(tables):
                # Filter empty rows
                rows = [r for r in table if any(c for c in r if c and str(c).strip())]
                if len(rows) >= 2:
                    found.append({
                        'page':         i + 1,
                        'table_index':  t_idx,
                        'rows':         len(rows),
                        'cols':         max(len(r) for r in rows),
                        'header_guess': rows[0],
                        'sample_row':   rows[1] if len(rows) > 1 else [],
                        'all_rows':     rows,
                    })
    return found


print('Scanning for tables (first 80 pages) …\n')
all_tables = {}
for name, path in PDFS.items():
    tables = find_tables(path)
    all_tables[name] = tables
    total, _, _ = page_classifications[name]
    print(f'  {name}: {len(tables)} table(s) found  (scanned {min(total,80)} of {total} pages)')

In [ ]:
MAX_TABLES_TO_SHOW = 12

for name, tables in all_tables.items():
    print(f"\n{'━'*65}")
    print(f'  📊 {name}  —  Table Inventory  ({len(tables)} tables)')
    print(f"{'━'*65}")
    if not tables:
        print('  (no tables detected in the first 80 pages)')
    for i, t in enumerate(tables[:MAX_TABLES_TO_SHOW]):
        header = [str(h)[:28] for h in t['header_guess']]
        sample = [str(v)[:28] for v in t['sample_row']]
        print(f"\n  Table {i+1}  (page {t['page']}, {t['rows']} rows × {t['cols']} cols)")
        print(f'    Header : {header}')
        print(f'    Row[1] : {sample}')
    if len(tables) > MAX_TABLES_TO_SHOW:
        print(f'\n  … and {len(tables) - MAX_TABLES_TO_SHOW} more tables not shown.')

---
## Question 5 — How is equipment information organized?

We scan for:
- **Section headings** (numbered titles)
- **Key-value pairs** (label: value)
- **Engineering units** (kW, RPM, kPa, °C …)
- **Domain-specific signals** (model numbers, refrigerant names, pressures, temperatures)

In [ ]:
PATTERNS = {
    'Model / Serial':   re.compile(r'(?:model|serial|s/?n|item|part)\s*[:#.\-]?\s*([A-Z0-9\-/]{3,})', re.I),
    'Numeric + Unit':   re.compile(r'\b(\d+\.?\d*)\s*(kW|kPa|bar|RPM|Hz|V|A|°C|°F|kg|L|cfm|m³|psig|psi|hp|kJ|MPa)\b', re.I),
    'Section Heading':  re.compile(r'^\s{0,4}(\d{1,2}\.?\d*\.?\d*\.?\s+[A-Z][^\n]{5,60})$', re.M),
    'Key-Value pair':   re.compile(r'^\s*([A-Za-z][\w\s/()]{2,40})\s*[:\-]\s*(.{1,80})$', re.M),
    'Temperature':      re.compile(r'(?:temperature|temp|suction|discharge)\s*[:#]?\s*(-?\d+\.?\d*\s*°?[CF])', re.I),
    'Pressure':         re.compile(r'(?:pressure|press)\s*[:#]?\s*(\d+\.?\d*\s*(?:kPa|bar|psi|psig|MPa))', re.I),
    'Capacity/Power':   re.compile(r'(?:capacity|power|rating|output|load)\s*[:#]?\s*(\d+\.?\d*\s*(?:kW|TR|tons|hp))', re.I),
    'Refrigerant':      re.compile(r'\b(R-?\d{2,4}[A-Z]?|ammonia|NH3|CO2|HFC|HCFC)\b', re.I),
}


def analyze_structure(pdf_path: Path, max_pages: int = 60):
    """Extract pattern hits and full text from up to max_pages pages."""
    pattern_hits = defaultdict(list)
    full_text = ''

    doc = pdfium.PdfDocument(str(pdf_path))
    n = min(len(doc), max_pages)
    for i in range(n):
        page_txt = doc[i].get_textpage().get_text_range()
        full_text += page_txt
        for label, pat in PATTERNS.items():
            for m in pat.finditer(page_txt):
                pattern_hits[label].append({'page': i + 1, 'match': m.group(0).strip()[:80]})
    doc.close()

    # De-duplicate section headings
    seen_headings = set()
    unique_headings = []
    for h in pattern_hits['Section Heading']:
        t = h['match'].strip()
        if t not in seen_headings:
            unique_headings.append(h)
            seen_headings.add(t)

    return pattern_hits, unique_headings, full_text


print('Analyzing document structure (first 60 pages each) …')
doc_analysis = {}
for name, path in PDFS.items():
    hits, headings, full_text = analyze_structure(path)
    doc_analysis[name] = (hits, headings, full_text)
    print(f'  ✅  {name}')
print('Done.')

In [ ]:
# Pattern hit summary table
print('Pattern Hit Summary  (first 60 pages per PDF)')
print('=' * 70)
header_line = f"  {'Pattern':<22}" + ''.join(f'  {n:>15}' for n in PDFS)
print(header_line)
print('  ' + '─' * (20 + 17 * len(PDFS)))

for label in PATTERNS:
    row = f'  {label:<22}'
    for name in PDFS:
        hits, _, _ = doc_analysis[name]
        row += f'  {len(hits[label]):>15,}'
    print(row)
print('=' * 70)

In [ ]:
# Section headings (table-of-contents proxy)
MAX_HEADINGS = 40

for name in PDFS:
    _, headings, _ = doc_analysis[name]
    print(f"\n{'━'*60}")
    print(f'  📑 {name}  —  Detected Section Headings  ({len(headings)} unique)')
    print(f"{'━'*60}")
    if not headings:
        print('  (none — headings may not follow a numbered format)')
    for h in headings[:MAX_HEADINGS]:
        print(f"    p.{h['page']:>4}   {h['match'].strip()}")
    if len(headings) > MAX_HEADINGS:
        print(f'    … and {len(headings)-MAX_HEADINGS} more')

In [ ]:
# Equipment signal samples: top 6 matches per signal type
SHOW = 6
SIGNAL_LABELS = ['Model / Serial', 'Refrigerant', 'Temperature',
                 'Pressure', 'Capacity/Power', 'Numeric + Unit']

for name in PDFS:
    hits, _, _ = doc_analysis[name]
    print(f"\n{'━'*60}")
    print(f'  🔎 {name}  —  Equipment Signal Samples')
    print(f"{'━'*60}")
    for label in SIGNAL_LABELS:
        items = hits[label][:SHOW]
        total = len(hits[label])
        if items:
            print(f'\n  [{label}]  ({total} total hits)')
            for item in items:
                print(f"    p.{item['page']:>3}  {item['match']}")

---
## Question 6 — Are the two PDFs structured similarly?

We compute a **structural fingerprint** for each PDF and compare them side-by-side.

In [ ]:
def fingerprint(name: str) -> dict:
    total_pages, page_data, overall = page_classifications[name]
    hits, headings, full_text        = doc_analysis[name]
    tables                           = all_tables[name]

    text_pages   = sum(1 for r in page_data if not r['is_scanned'])
    scanned_pages = sum(1 for r in page_data if r['is_scanned'])

    return {
        'total_pages':            total_pages,
        'classification':         overall,
        'text_pages (sampled)':   text_pages,
        'scanned_pages (sampled)': scanned_pages,
        'section_headings':       len(headings),
        'tables_found':           len(tables),
        'kv_pairs_found':         len(hits['Key-Value pair']),
        'numeric_unit_hits':      len(hits['Numeric + Unit']),
        'refrigerant_mentions':   len(hits['Refrigerant']),
        'chars_in_60pg_sample':   len(full_text),
    }


fps = {name: fingerprint(name) for name in PDFS}

print('Structural Fingerprint Comparison')
print('=' * 68)
keys = list(list(fps.values())[0].keys())
print(f"  {'Metric':<30}" + ''.join(f'  {n:>16}' for n in PDFS))
print('  ' + '─' * (28 + 18 * len(PDFS)))
for k in keys:
    row = f'  {k:<30}'
    for name in PDFS:
        v = fps[name][k]
        row += f'  {str(v):>16}' if isinstance(v, str) else f'  {v:>16,}'
    print(row)
print('=' * 68)

In [ ]:
# Qualitative similarity verdict
names = list(PDFS.keys())
a, b = names[0], names[1]

same_class   = fps[a]['classification'] == fps[b]['classification']
h_a, h_b     = fps[a]['section_headings'], fps[b]['section_headings']
heading_ratio = min(h_a, h_b) / max(h_a + h_b, 1)
both_tables  = fps[a]['tables_found'] > 0 and fps[b]['tables_found'] > 0

print('\n📋 Structural Similarity Assessment')
print('=' * 60)
print(f"  Same text/scan classification? : {'✅ Yes' if same_class else '❌ No  (' + fps[a]['classification'] + ' vs ' + fps[b]['classification'] + ')'}")
print(f"  Both contain tables?           : {'✅ Yes' if both_tables else '❌ No'}")
print(f'  Heading-count similarity ratio : {heading_ratio:.2f}  (1.0 = same count)')
print()

if same_class and both_tables and heading_ratio > 0.3:
    verdict = 'STRUCTURALLY SIMILAR'
    advice  = 'A unified extraction pipeline should work for both PDFs.'
elif same_class:
    verdict = 'SAME TYPE, DIFFERENT INTERNAL LAYOUT'
    advice  = 'Separate parsing strategies may be needed per document.'
else:
    verdict = 'STRUCTURALLY DIFFERENT'
    advice  = 'Different parsing approaches are required for each PDF.'

print(f'  ➡  Verdict: {verdict}')
print(f'     {advice}')
print('=' * 60)

---
## Question 7 — What fields do we need to extract?

We combine two approaches:
1. **Data-driven:** rank the most-frequent key strings found in the documents.
2. **Domain-driven:** a curated schema based on industrial refrigeration equipment standards.

In [ ]:
# ── 7a. Data-driven: top KV keys from both PDFs ───────────────────────────
def extract_kv_keys(full_text: str, top_n: int = 40) -> list:
    kv_pat = re.compile(r'^\s*([A-Za-z][\w\s/()]{2,35})\s*[:\-]\s*.+$', re.M)
    keys = []
    for m in kv_pat.finditer(full_text):
        key = m.group(1).strip().lower()
        if len(key.split()) <= 6:   # skip sentence-like lines
            keys.append(key)
    return Counter(keys).most_common(top_n)


combined = Counter()
for name in PDFS:
    _, _, full_text = doc_analysis[name]
    doc_keys = dict(extract_kv_keys(full_text))
    combined += Counter(doc_keys)

    print(f"  {'─'*50}")
    print(f'  Top 25 keys in  {name}')
    print(f"  {'─'*50}")
    for k, v in list(doc_keys.items())[:25]:
        print(f'    {v:>5}×  {k}')

print(f"\n  {'─'*50}")
print('  Combined Top 30 Keys  (both PDFs)')
print(f"  {'─'*50}")
for k, v in combined.most_common(30):
    print(f'    {v:>5}×  {k}')

In [ ]:
# ── 7b. Domain-driven: curated extraction schema ─────────────────────────
EXTRACTION_SCHEMA = {
    'Identification': [
        ('model_number',        'Model or product number of the unit'),
        ('serial_number',       'Unique serial number'),
        ('item_number',         'Item or tag number'),
        ('part_number',         'Spare part identifier'),
        ('equipment_tag',       'Plant / P&ID tag'),
        ('manufacturer',        'OEM name'),
        ('document_title',      'Title of the manual / datasheet'),
        ('document_revision',   'Revision or version code'),
        ('document_date',       'Issue or revision date'),
    ],
    'Refrigeration Circuit': [
        ('refrigerant_type',         'Refrigerant designation e.g. R-717, R-404A'),
        ('refrigerant_charge_kg',    'Total refrigerant charge in kg'),
        ('compressor_type',          'Screw / reciprocating / scroll / centrifugal'),
        ('compressor_model',         'Compressor model number'),
        ('cooling_capacity_kW',      'Net cooling capacity in kW'),
        ('suction_temp_degC',        'Suction / evaporation temperature °C'),
        ('discharge_temp_degC',      'Discharge / condensing temperature °C'),
        ('suction_pressure_kPa',     'Low-side pressure kPa(g)'),
        ('discharge_pressure_kPa',   'High-side pressure kPa(g)'),
        ('operating_speed_RPM',      'Compressor shaft speed RPM'),
        ('COP',                      'Coefficient of performance'),
    ],
    'Electrical': [
        ('supply_voltage_V',        'Supply voltage e.g. 415 V'),
        ('frequency_Hz',            'Supply frequency Hz'),
        ('power_input_kW',          'Total electrical power input kW'),
        ('full_load_current_A',     'FLA in amperes'),
        ('motor_model',             'Motor model number'),
        ('motor_frame',             'IEC / NEMA frame size'),
        ('IP_rating',               'Ingress protection class'),
    ],
    'Physical': [
        ('weight_operating_kg',  'Operating weight kg'),
        ('weight_shipping_kg',   'Shipping weight kg'),
        ('dimensions_L_mm',      'Overall length mm'),
        ('dimensions_W_mm',      'Overall width mm'),
        ('dimensions_H_mm',      'Overall height mm'),
        ('oil_type',             'Lubricant type / grade'),
        ('oil_charge_L',         'Oil charge volume litres'),
    ],
    'Operating Limits': [
        ('ambient_temp_min_degC',  'Minimum ambient temperature °C'),
        ('ambient_temp_max_degC',  'Maximum ambient temperature °C'),
        ('design_pressure_kPa',   'Design pressure kPa(g)'),
        ('test_pressure_kPa',     'Test / proof pressure kPa(g)'),
        ('min_op_temp_degC',      'Minimum operating temp °C'),
        ('max_op_temp_degC',      'Maximum operating temp °C'),
    ],
    'Maintenance': [
        ('oil_change_interval_h',      'Oil change interval hours'),
        ('filter_change_interval',     'Filter replacement schedule'),
        ('overhaul_interval_h',        'Major overhaul interval hours'),
        ('alarm_high_pressure_kPa',    'High-pressure alarm setpoint kPa'),
        ('alarm_low_pressure_kPa',     'Low-pressure alarm setpoint kPa'),
        ('alarm_high_temp_degC',       'High-temperature alarm setpoint °C'),
        ('safety_cutout_pressure_kPa', 'High-pressure safety cutout kPa'),
    ],
}


print('\n' + '=' * 70)
print('  📋 RECOMMENDED EXTRACTION SCHEMA')
print('  (curated for industrial refrigeration equipment manuals)')
print('=' * 70)

total_fields = 0
for category, fields in EXTRACTION_SCHEMA.items():
    print(f"\n  🏷  {category}  ({len(fields)} fields)")
    print(f"  {'Field Name':<35} Description")
    print(f"  {'─'*34}  {'─'*30}")
    for field_name, description in fields:
        print(f"  {field_name:<35} {description}")
    total_fields += len(fields)

print(f"\n  Total fields: {total_fields}")
print('=' * 70)

---
## Summary Report — Consolidated Findings

In [ ]:
SEP = '=' * 70

print(SEP)
print('  CONSOLIDATED FINDINGS  —  PDF EXPLORATION REPORT')
print(SEP)

for name in PDFS:
    total_pages, _, overall = page_classifications[name]
    fp = fps[name]
    tables = all_tables[name]
    hits, headings, _ = doc_analysis[name]
    char_count = fp['chars_in_60pg_sample']

    print(f"""
  ┌─ {name} {'─'*(54-len(name))}┐
  │  Q1  Type                : {overall:<28}│
  │  Q2  Total pages         : {total_pages:<28,}│
  │  Q3  Text extractable?   : {'Yes — rich text content' if char_count > 5000 else 'Minimal / None':<28}│
  │      Chars (60-pg sample): {char_count:<28,}│
  │  Q4  Tables found        : {len(tables):<28}│
  │  Q5  Section headings    : {len(headings):<28}│
  │      KV pairs            : {len(hits['Key-Value pair']):<28,}│
  │      Numeric+Unit hits   : {len(hits['Numeric + Unit']):<28,}│
  └{'─'*57}┘""")

print(f"""
  Q6  Structural similarity:
      Verdict  : {verdict}
      Advice   : {advice}

  Q7  Extraction schema:
      Categories: {', '.join(EXTRACTION_SCHEMA.keys())}
      Total fields defined: {total_fields}
""")
print(SEP)
print('  ✅  Exploration complete.')
print('  ➡  Next step: 02_extraction_pipeline.ipynb')
print(SEP)

---
*Notebook: `01_pdf_exploration.ipynb`  
Project: Industrial Equipment Data Extraction*